## Módulo 5. Spark SQL en Databricks

## Introducción

¡Bienvenido al módulo Spark SQL en Databricks! A lo largo de este curso, nos enfocaremos en aplicar conceptos esenciales de SQL utilizando ejemplos reales y ejercicios prácticos. Para ello, trabajaremos con dos tablas del esquema bakehouse:

* **sales_transactions:** contiene los registros detallados de cada venta realizada, con información como fecha, producto, cantidad y monto total.
* **sales_franchises:** almacena los datos de las franquicias, incluyendo ubicación y detalles relevantes de cada punto de venta.

En este curso, analizaremos cada una de estas tablas de manera individual. Aprenderás cómo explorar su estructura, consultar su información, y extraer datos valiosos mediante consultas SQL. Te guiaremos paso a paso para que domines las técnicas fundamentales y puedas resolver problemas reales de análisis de datos.

## Día 1. Tabla de ventas

**1. ¿Qué campos tiene la tabla?**

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions limit 10;

In [0]:
%sql
DESCRIBE samples.bakehouse.sales_transactions;

**2. ¿Cuántos registros tiene la tabla?**

In [0]:
%sql
SELECT count(*) as total_registros FROM samples.bakehouse.sales_transactions;

**3. ¿Qué histórico hay de información? (dias, meses, años)**

In [0]:
%sql
SELECT MIN(dateTime) as fecha_min, MAX(dateTime) as fecha_max
FROM samples.bakehouse.sales_transactions;

**4. ¿Qué contiene el campo "product"?**

In [0]:
%sql
SELECT DISTINCT(product) as productos 
FROM samples.bakehouse.sales_transactions;

**5. ¿Y si quiero analizar un producto en particular?**

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions
WHERE product = 'Pearly Pies'
LIMIT 10;

**6. ¿Y si solo quiero ver las transacciones con mas ventas?**

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions
WHERE product = 'Pearly Pies'
AND quantity > 10;

**7. ¿Que se vendió en un solo día?**

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions
WHERE DATE(dateTime) = '2024-05-01'            
LIMIT 10;


**8. ¿Y si quiero analizar varios días?**

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions
WHERE DATE(dateTime) BETWEEN '2024-05-01' AND '2024-05-05'
LIMIT 10;

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions
WHERE DATE(dateTime) IN ('2024-05-01', '2024-05-05', '2024-05-07')
LIMIT 10;

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_transactions
WHERE DATE(dateTime) = (
  SELECT MIN(DATE(dateTime))
  FROM samples.bakehouse.sales_transactions
)
LIMIT 10;

**9. ¿Cuántos productos se venden por día?**

In [0]:
%sql
SELECT DATE(dateTime) as fecha, SUM(quantity) as n_productos 
FROM samples.bakehouse.sales_transactions
GROUP BY DATE(dateTime)
ORDER BY DATE(dateTime);

Databricks visualization. Run in Databricks to view.

**10. Quiero guardar el resumen anterior para futuros análisis**

In [0]:
%sql
CREATE OR REPLACE TABLE workspace.default.daily_products AS
  SELECT DATE(dateTime) as fecha, SUM(quantity) as n_productos
  FROM samples.bakehouse.sales_transactions
  GROUP BY DATE(dateTime)
  ORDER BY DATE(dateTime);

**NOTA:** <br>
En el ambiente productivo solo se deben crear **tablas finales** que se vayan a compartir con otros usuarios/áreas o que vayan a alimentar tableros de PowerBI.

Además, se debe crear siguiendo la nomenclatura establecida siguiendo los lineamientos de **Dominios y Arquitectura Medallón.**

#### Ejercicio 1
Crear una consulta SQL que muestre, para cada producto, el total de ventas acumuladas, la venta más alta y la más baja registrada.

La consulta debe devolver una tabla con las siguientes columnas:
* product
* total_ventas
* venta_maxima
* venta_minima


In [0]:
%sql
SELECT
  product,
  SUM(totalPrice) AS total_ventas,
  MAX(totalPrice) AS venta_maxima,
  MIN(totalPrice) AS venta_minima
FROM samples.bakehouse.sales_transactions
GROUP BY product
ORDER BY product;


#### Ejercicio 2
Crear una consulta SQL que muestre, por día, el total de ventas acumuladas y el promedio de ventas, para los días posteriores al 10 de mayo.

La consulta debe devolver una tabla ordenada de menor a mayor fecha con las siguientes columnas:
* fecha
* total_ventas
* promedio_ventas



In [0]:
%sql
SELECT
 DATE(dateTime) AS fecha,
  SUM(quantity) AS total_ventas,
  round(AVG(quantity),2) AS promedio_ventas
FROM samples.bakehouse.sales_transactions
WHERE DATE(dateTime) > '2024-05-10'
GROUP BY DATE(dateTime)
ORDER BY fecha ASC;

### Ejercicio 3: Análisis de Ventas de Productos Seleccionados

**Objetivo:**
Crear una consulta SQL que muestre el resumen diario de ventas (cantidad, valor total y promedio) para productos específicos dentro de un rango de fechas determinado.

**Instrucciones:**

**1. Base de datos y tabla:**

Utiliza la tabla samples.bakehouse.sales_transactions. Esta tabla contiene información sobre las transacciones de ventas, incluyendo la fecha y hora de la transacción (dateTime), el producto vendido (product), la cantidad de unidades vendidas (quantity) y el precio total de la venta (totalPrice).

**2. Filtros requeridos:**
  * Considera únicamente las transacciones de los siguientes productos: 'Tokyo Tidbits' y 'Pearly Pies'.
  * Limita los resultados a las transacciones realizadas entre el 1 de mayo de 2024 y el 10 de mayo de 2024 (ambas fechas incluidas).

**3. Agrupación de datos:**
  * Agrupa los resultados por fecha y por producto.
  * Para cada grupo, debes calcular:
    - El total de unidades vendidas (cantidad)
    - El total de ventas en dinero (total_ventas)
    - El promedio de ventas en dinero (promedio_ventas)

**4. Formato de salida:**
  * La consulta debe mostrar las siguientes columnas:
    - fecha: la fecha de la transacción (sin la hora).
    - product: nombre del producto.
    - cantidad: suma total de las unidades vendidas por producto y fecha.
    - total_ventas: suma total del valor de ventas por producto y fecha.
    - promedio_ventas: promedio de ventas por producto y fecha.

**5. Ordenamiento:**
  * Ordena los resultados por fecha y por producto, ambos en orden ascendente.

In [0]:
%sql
SELECT
DATE(dateTime) AS fecha
,product
,SUM(quantity) AS cantidad
,SUM(totalPrice) as total_ventas
,AVG(totalPrice) as promedio_ventas
FROM
samples.bakehouse.sales_transactions
WHERE
DATE(dateTime) >= '2024-05-01' AND
DATE(dateTime) <= '2024-05-10' AND
product IN ('Tokyo Tidbits', 'Pearly Pies')
GROUP BY
DATE(dateTime)
,product
ORDER BY
DATE(dateTime)
,product
;


Databricks visualization. Run in Databricks to view.